In [1]:
# all_model_comparison_linear.py
#
# Evaluate:
#   - kNN / SVM / RF on:
#       (a) original traditional_flat.npz test split
#       (b) RadioML2016_10A 70/15/15 test split
#   - 1D CNN / RNN / Hybrid on RadioML2016_10A 70/15/15 test split
# and save summary plots to:
#   /content/drive/MyDrive/amc_runs/final_models_comparison/summary

import os
import pickle
import random
import csv
import json


import numpy as np
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------
# 0. Paths / config
# ---------------------------------------------------------------------

# If you're in Colab, mount Drive before running:
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except ImportError:
    print("google.colab not available; assuming local filesystem.")

# Original traditional NPZ (flattened features + saved split)
NPZ_PATH = "/content/drive/MyDrive/AMC_datasets/RadioML2016_10A_traditional_flat.npz"
TRAD_SNR_JSON = "/content/drive/MyDrive/amc_runs/traditional_models/summary.json"


# Dict-style RadioML 2016.10A dataset (mod, SNR) -> array
DATASET_PATH = "/content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl"

# Root folder for all final comparison models
COMPARISON_ROOT = "/content/drive/MyDrive/amc_runs/final_model_comparison"
TRAD_BASE = COMPARISON_ROOT  # knn/svm/rf live directly under this

# Folder to save summary figures / CSV
SUMMARY_ROOT = "/content/drive/MyDrive/amc_runs/final_model_comparison/summary2"

BATCH_SIZE = 256
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------------------

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

ensure_dir(SUMMARY_ROOT)

# ---------------------------------------------------------------------
# 2. Dataset utilities
# ---------------------------------------------------------------------

def load_radioml_dataset(path: str):
    """
    Load a RadioML-style dataset where the pickle is a dict with keys like:
        (modulation_str, snr_int) -> array of shape (N_per_pair, 2, 128)
    Returns:
        X   : (N, 2, 128)
        y   : (N,) array of modulation labels (strings)
        snr : (N,) array of SNR values
    """
    with open(path, "rb") as f:
        try:
            data = pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0)
            data = pickle.load(f, encoding="latin1")

    if not isinstance(data, dict):
        raise ValueError(f"Expected dict-style RadioML pickle at {path}")

    X_list, y_list, snr_list = [], [], []
    for key, arr in data.items():
        if not (isinstance(key, tuple) and len(key) == 2):
            continue
        mod, snr = key
        arr = np.asarray(arr)
        if arr.ndim != 3:
            continue
        n = arr.shape[0]
        X_list.append(arr)
        y_list.extend([mod] * n)
        snr_list.extend([snr] * n)

    if not X_list:
        raise ValueError(f"No (mod, SNR)->3D entries found in {path}")

    X = np.concatenate(X_list, axis=0)  # (N, 2, 128) or (N, 128, 2)
    if X.shape[1:] == (128, 2):
        X = np.transpose(X, (0, 2, 1))  # (N, 2, 128)

    y = np.array(y_list)
    snr = np.array(snr_list)

    print(f"Loaded {path} in (mod, SNR) -> (N, 2, 128) format:")
    print("  X shape:", X.shape)
    print("  y shape:", y.shape, "dtype:", y.dtype)
    print("  snr shape:", snr.shape, "dtype:", snr.dtype)
    return X, y, snr


def make_splits_70_15_15(X, y, snr, seed: int = 42):
    """Return (train, val, test) splits with stratification on y."""
    X_tr, X_tmp, y_tr, y_tmp, snr_tr, snr_tmp = train_test_split(
        X, y, snr, test_size=0.30, stratify=y, random_state=seed
    )
    X_val, X_te, y_val, y_te, snr_val, snr_te = train_test_split(
        X_tmp, y_tmp, snr_tmp, test_size=0.50, stratify=y_tmp, random_state=seed
    )
    return (X_tr, y_tr, snr_tr,
            X_val, y_val, snr_val,
            X_te, y_te, snr_te)


def to_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size: int = 256):
    """Create PyTorch loaders for deep models."""
    def make_loader(X, y, shuffle: bool):
        Xt = torch.from_numpy(X).float()
        yt = torch.from_numpy(y).long()
        ds = TensorDataset(Xt, yt)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

    train_loader = make_loader(X_tr,  y_tr,  shuffle=True)
    val_loader   = make_loader(X_val, y_val, shuffle=False)
    test_loader  = make_loader(X_te,  y_te,  shuffle=False)
    return train_loader, val_loader, test_loader

# ---------------------------------------------------------------------
# 3. Traditional model helpers (GPU-safe)
# ---------------------------------------------------------------------
def load_trad_test_acc_from_json(json_path: str):
    """
    Load overall test accuracy for kNN / SVM / RF from the
    traditional_models summary.json file.

    Returns:
        dict: {"kNN": acc_pct, "SVM": acc_pct, "RF": acc_pct}
              where acc_pct is in percentage (0–100).
    """
    if not os.path.exists(json_path):
        print(f"[WARN] Traditional summary JSON not found at {json_path}")
        return {}

    with open(json_path, "r") as f:
        data = json.load(f)

    results = {}
    for key, label in [("knn", "kNN"), ("svm", "SVM"), ("rf", "RF")]:
        entry = data.get(key, {})
        test_acc = entry.get("test_accuracy", None)
        if test_acc is None:
            continue

        # summary.json stores fractional accuracy (0–1), convert → %
        results[label] = float(test_acc) * 100.0

    print("[INFO] Loaded traditional test accuracies from JSON:", results)
    return results


def load_trad_snr_curves(json_path: str):
    """
    Load precomputed SNR→accuracy curves for traditional models
    from summary.json produced by the original traditional_models run.
    Returns: dict like {"kNN": (snr_np, acc_pct_np), ...}
    """
    if not os.path.exists(json_path):
        print(f"[WARN] Traditional SNR JSON not found at {json_path}")
        return {}

    with open(json_path, "r") as f:
        data = json.load(f)

    out = {}
    for key, label in [("knn", "kNN"), ("svm", "SVM"), ("rf", "RF")]:
        entry = data.get(key, {})
        sc = entry.get("snr_curve", None)
        if sc is None:
            continue
        snr = np.array(sc["snr"], dtype=float)
        acc = np.array(sc["accuracy"], dtype=float) * 100.0  # convert fraction→%
        out[label] = (snr, acc)
    return out

def _try_import_cuml():
    """Best-effort import of cuML + CuPy stack; return None on failure."""
    try:
        import cupy as cp  # type: ignore
        from cuml.neighbors import KNeighborsClassifier as cuKNN  # noqa: F401
        from cuml.svm import SVC as cuSVC  # noqa: F401
        from cuml.ensemble import RandomForestClassifier as cuRF  # noqa: F401
        return {"cp": cp, "cuKNN": cuKNN, "cuSVC": cuSVC, "cuRF": cuRF}
    except Exception:
        return None


def to_numpy(x):
    """Convert CuPy arrays to NumPy when needed."""
    try:
        import cupy as cp  # type: ignore
        if isinstance(x, cp.ndarray):
            return cp.asnumpy(x)
    except Exception:
        pass
    return x


def predict_trad(est, X):
    """
    Predict with a traditional model that may be:
      - plain sklearn estimator,
      - ('gpu_model', cuML_model),
      - ('scaler+gpu_model', scaler, cuML_model)
    """
    if not isinstance(est, tuple):
        return est.predict(X)

    tag = est[0]
    gpu = _try_import_cuml()
    if gpu is None:
        raise RuntimeError("cuML/CuPy not available but GPU model was loaded.")
    cp = gpu["cp"]

    if tag == "gpu_model":
        model = est[1]
        return to_numpy(model.predict(cp.asarray(X)))
    elif tag == "scaler+gpu_model":
        _, scaler, model = est
        Xs = scaler.transform(X)
        return to_numpy(model.predict(cp.asarray(Xs)))
    else:
        raise ValueError(f"Unknown estimator wrapper tag: {tag}")

# ---------------------------------------------------------------------
# 4. Torch models: CNN / RNN / Hybrid
# ---------------------------------------------------------------------

def conv_block(in_channels: int,
               out_channels: int,
               kernel_size: int = 3,
               stride: int = 1,
               padding: int = 1):
    return nn.Sequential(
        nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        ),
        nn.BatchNorm1d(out_channels),
        nn.ReLU(inplace=True),
    )


class AMC1DCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(2, 32),   conv_block(32, 32),   nn.MaxPool1d(2),  # 128→64
            conv_block(32, 64),  conv_block(64, 64),  nn.MaxPool1d(2),  # 64→32
            conv_block(64, 128), conv_block(128, 128), nn.MaxPool1d(2), # 32→16
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if getattr(m, "bias", None) is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)


class AMCRNN(nn.Module):
    def __init__(self,
                 num_classes: int,
                 cell: str = "gru",
                 hidden_size: int = 256,
                 num_layers: int = 2,
                 bidirectional: bool = False,
                 dropout: float = 0.3):
        super().__init__()
        input_dim = 2
        self.cell = cell.lower()
        self.bidirectional = bidirectional

        rnn_kwargs = dict(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        if self.cell == "gru":
            self.rnn = nn.GRU(**rnn_kwargs)
        else:
            self.rnn = nn.LSTM(**rnn_kwargs)

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)   # (B, 2, 128) → (B, 128, 2)
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        logits = self.fc(last)
        return logits


class CNNRNNEnsemble(nn.Module):
    def __init__(self,
                 cnn_model: nn.Module,
                 rnn_model: nn.Module,
                 w_cnn: float = 0.5,
                 w_rnn: float = 0.5):
        super().__init__()
        self.cnn = cnn_model
        self.rnn = rnn_model
        self.w_cnn = w_cnn
        self.w_rnn = w_rnn

    def forward(self, x):
        logits_cnn = self.cnn(x)
        logits_rnn = self.rnn(x)
        return self.w_cnn * logits_cnn + self.w_rnn * logits_rnn

# ---------------------------------------------------------------------
# 5. Loading / evaluating deep models
# ---------------------------------------------------------------------

@torch.no_grad()
def eval_torch_model(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.numel()
    return 100.0 * correct / max(1, total)


@torch.no_grad()
def get_torch_preds(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds = []
    for xb, _ in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds, axis=0)


def load_cnn(num_classes: int) -> nn.Module:
    ckpt = os.path.join(COMPARISON_ROOT, "cnn", "best_model.pth")
    model = AMC1DCNN(num_classes=num_classes).to(device)
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Loaded CNN from:", ckpt)
    return model


def load_rnn(num_classes: int) -> nn.Module:
    ckpt = os.path.join(COMPARISON_ROOT, "rnn", "best_model.pth")
    model = AMCRNN(
        num_classes=num_classes,
        cell="gru",
        hidden_size=256,
        num_layers=2,
        bidirectional=False,
        dropout=0.3,
    ).to(device)
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Loaded RNN from:", ckpt)
    return model


def load_hybrid(cnn: nn.Module, rnn: nn.Module) -> nn.Module:
    ckpt_path = os.path.join(COMPARISON_ROOT, "hybrid", "best_model.pth")
    if os.path.exists(ckpt_path):
        bundle = torch.load(ckpt_path, map_location=device)
        w_cnn = float(bundle.get("w_cnn", 0.5))
        w_rnn = float(bundle.get("w_rnn", 0.5))
        hybrid = CNNRNNEnsemble(cnn, rnn, w_cnn=w_cnn, w_rnn=w_rnn).to(device)
        hybrid.load_state_dict(bundle["hybrid_state_dict"])
        hybrid.eval()
        print(
            f"Loaded Hybrid from {ckpt_path} with "
            f"w_cnn={w_cnn:.2f}, w_rnn={w_rnn:.2f}"
        )
    else:
        hybrid = CNNRNNEnsemble(cnn, rnn, w_cnn=0.5, w_rnn=0.5).to(device)
        print("Hybrid checkpoint not found; using 0.5/0.5 ensemble.")
    return hybrid

# ---------------------------------------------------------------------
# 6. Evaluation routines
# ---------------------------------------------------------------------

def eval_trad_on_npz():
    """Evaluate traditional models on the ORIGINAL traditional_flat.npz test split."""
    print("\n=== Loading original traditional_flat NPZ ===")
    data_npz = np.load(NPZ_PATH, allow_pickle=True)
    X_test_flat = data_npz["X_test"]
    y_test = data_npz["y_test"]

    print("X_test_flat shape (NPZ):", X_test_flat.shape)
    print("y_test shape (NPZ):", y_test.shape)

    results = {}
    for key, label in [("knn", "kNN"), ("svm", "SVM"), ("rf", "RF")]:
        model_path = os.path.join(TRAD_BASE, key, "model.pkl")
        if not os.path.exists(model_path):
            print(f"[WARN] {model_path} not found; skipping {label}.")
            continue

        print(f"\n=== Reloading {label} from {model_path} ===")
        est = joblib.load(model_path)
        y_pred = predict_trad(est, X_test_flat)
        acc_pct = 100.0 * accuracy_score(y_test, y_pred)
        results[label] = acc_pct
        print(f"{label} accuracy on ORIGINAL NPZ TEST split: {acc_pct:.2f}%")

    print("\n=== Traditional models – ORIGINAL NPZ TEST split (%) ===")
    for k, v in results.items():
        print(f"{k:8s}: {v:5.2f}")
    return results


def eval_trad_on_2016_test(X_te, y_te, return_preds: bool = False):
    """Evaluate traditional models on the 2016 70/15/15 test split."""
    X_te_flat = X_te.reshape(X_te.shape[0], -1)
    print("\nX_te_flat shape (2016 TEST):", X_te_flat.shape)

    results = {}
    preds_dict = {}
    for key, label in [("knn", "kNN"), ("svm", "SVM"), ("rf", "RF")]:
        model_path = os.path.join(TRAD_BASE, key, "model.pkl")
        if not os.path.exists(model_path):
            print(f"[WARN] {model_path} not found; skipping {label}.")
            continue

        print(f"\n=== Reloading {label} from {model_path} ===")
        est = joblib.load(model_path)
        y_pred = predict_trad(est, X_te_flat)
        acc_pct = 100.0 * accuracy_score(y_te, y_pred)
        results[label] = acc_pct
        preds_dict[label] = y_pred
        print(f"{label} accuracy on 2016 TEST split: {acc_pct:.2f}%")

    print("\n=== 2016 – Traditional models on 70/15/15 TEST split (%) ===")
    for k, v in results.items():
        print(f"{k:8s}: {v:5.2f}")

    if return_preds:
        return results, preds_dict
    return results


def eval_deep_on_2016_test(num_classes,
                           X_tr, y_tr,
                           X_val, y_val,
                           X_te, y_te,
                           return_preds: bool = False):
    """Evaluate CNN, RNN, Hybrid on the 2016 70/15/15 test split."""
    train_loader, val_loader, test_loader = to_loaders(
        X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size=BATCH_SIZE
    )

    cnn = load_cnn(num_classes)
    rnn = load_rnn(num_classes)
    hybrid = load_hybrid(cnn, rnn)

    results = {
        "1D CNN": eval_torch_model(cnn, test_loader),
        "RNN":     eval_torch_model(rnn, test_loader),
        "Hybrid":  eval_torch_model(hybrid, test_loader),
    }

    print("\n=== 2016 – Deep models on 70/15/15 TEST split (%) ===")
    for k, v in results.items():
        print(f"{k:8s}: {v:5.2f}")

    preds = None
    if return_preds:
        preds = {
            "1D CNN": get_torch_preds(cnn, test_loader),
            "RNN":    get_torch_preds(rnn, test_loader),
            "Hybrid": get_torch_preds(hybrid, test_loader),
        }

    if return_preds:
        return results, preds
    return results

# ---------------------------------------------------------------------
# 7. Plotting helpers (saved to SUMMARY_ROOT)
# ---------------------------------------------------------------------

def save_accuracy_bar(trad_results, deep_results, out_path: str):
    labels = []
    values = []
    for name in ["kNN", "SVM", "RF"]:
        if name in trad_results:
            labels.append(name)
            values.append(trad_results[name])
    for name in ["1D CNN", "RNN", "Hybrid"]:
        if name in deep_results:
            labels.append(name)
            values.append(deep_results[name])

    if not labels:
        print("No accuracies to plot for bar chart.")
        return

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x, values)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title("RadioML 2016.10A – 70/15/15 TEST accuracy by model")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print("Saved bar chart to:", out_path)


def save_confusion_matrix(y_true, y_pred, class_names, title: str, out_path: str):
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(invalid="ignore"):
        cm_norm = cm.astype("float") / np.clip(cm.sum(axis=1, keepdims=True), 1e-9, None)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, interpolation="nearest")
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print("Saved confusion matrix to:", out_path)


def compute_snr_curve(y_true, y_pred, snr_arr):
    uniq_snr = np.sort(np.unique(snr_arr))
    acc = []
    for s in uniq_snr:
        idx = (snr_arr == s)
        if idx.sum() == 0:
            acc.append(np.nan)
        else:
            acc.append((y_pred[idx] == y_true[idx]).mean() * 100.0)
    return uniq_snr, np.array(acc)


def save_snr_curves(y_true, snr_te, deep_preds_dict, trad_snr_dict, out_path: str):
    """
    Plot accuracy vs SNR.
      - trad_snr_dict: precomputed curves from summary.json (already in %)
      - deep_preds_dict: logits-based predictions from this run
    """
    fig, ax = plt.subplots(figsize=(8, 4))

    # 1) Traditional curves (from old run)
    for name in ["kNN", "SVM", "RF"]:
        if name not in trad_snr_dict:
            continue
        snr_vals, acc_vals = trad_snr_dict[name]
        ax.plot(
            snr_vals,
            acc_vals,
            marker="x",
            linestyle="--",
            label=f"{name} (trad)",
        )

    # 2) Deep curves (from this run)
    for name in ["1D CNN", "RNN", "Hybrid"]:
        if name not in deep_preds_dict:
            continue
        y_pred = deep_preds_dict[name]
        snr_vals = np.sort(np.unique(snr_te))
        acc_vals = []
        for s in snr_vals:
            idx = (snr_te == s)
            if idx.sum() == 0:
                acc_vals.append(np.nan)
            else:
                acc_vals.append((y_pred[idx] == y_true[idx]).mean() * 100.0)
        acc_vals = np.array(acc_vals)
        ax.plot(
            snr_vals,
            acc_vals,
            marker="o",
            label=f"{name} (deep)",
        )

    ax.set_xlabel("SNR (dB)")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("RadioML 2016.10A – Test accuracy vs SNR (trad + deep)")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print("Saved SNR curves (trad + deep) to:", out_path)



def save_accuracy_csv(trad_results, deep_results, out_path: str):
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["model", "family", "test_accuracy_2016_70_15_15"])
        for k, v in trad_results.items():
            writer.writerow([k, "traditional", f"{v:.4f}"])
        for k, v in deep_results.items():
            writer.writerow([k, "deep", f"{v:.4f}"])
    print("Saved accuracy CSV to:", out_path)

# ---------------------------------------------------------------------
# 8. Main
# ---------------------------------------------------------------------

def main():
    # 1) Traditional models on original NPZ split (printed only)
    _ = eval_trad_on_npz()


    trad_snr_dict = load_trad_snr_curves(TRAD_SNR_JSON)

    # 2) Load RadioML 2016.10A dict-style dataset
    print("\n=== Loading RadioML 2016.10A dataset ===")
    X16, y16_raw, snr16 = load_radioml_dataset(DATASET_PATH)
    le16 = LabelEncoder()
    y16 = le16.fit_transform(y16_raw)
    num_classes16 = len(le16.classes_)
    class_names = list(le16.classes_)
    print("2016 mods:", class_names, "| num_classes =", num_classes16)

    # 3) Make 70/15/15 split
    (X16_tr, y16_tr, snr16_tr,
     X16_val, y16_val, snr16_val,
     X16_te, y16_te, snr16_te) = make_splits_70_15_15(X16, y16, snr16, seed=SEED)

    print("\n=== 70/15/15 split (2016) ===")
    print("Train:", X16_tr.shape, y16_tr.shape)
    print("Val  :", X16_val.shape, y16_val.shape)
    print("Test :", X16_te.shape, y16_te.shape)

    # 4) Traditional models on 2016 70/15/15 test split
    trad_2016_results, trad_2016_preds = eval_trad_on_2016_test(
        X16_te, y16_te, return_preds=True
    )

    # Use JSON-based accuracies for final CSV / bar chart,
    # but keep the deep-model pipeline untouched.
    trad_summary_results = load_trad_test_acc_from_json(TRAD_SNR_JSON)
    if not trad_summary_results:
        # Fallback: if JSON missing/broken, use the freshly computed 2016 results
        trad_summary_results = trad_2016_results

    # 5) Deep models on 2016 70/15/15 test split
    deep_2016_results, deep_2016_preds = eval_deep_on_2016_test(
        num_classes16,
        X16_tr, y16_tr,
        X16_val, y16_val,
        X16_te, y16_te,
        return_preds=True,
    )


    # 6) Final consolidated summary print
    print("\n=== Final comparison on RadioML 2016.10A (70/15/15 TEST) ===")
    for label in ["kNN", "SVM", "RF"]:
        if label in trad_summary_results:
            print(f"{label:8s} (trad, from JSON): {trad_summary_results[label]:5.2f}%")
        elif label in trad_2016_results:
            # Fallback if JSON was missing that model
            print(f"{label:8s} (trad, 2016 run):  {trad_2016_results[label]:5.2f}%")
        else:
            print(f"{label:8s} (trad):  N/A")

    for label in ["1D CNN", "RNN", "Hybrid"]:
        if label in deep_2016_results:
            print(f"{label:8s} (deep): {deep_2016_results[label]:5.2f}%")
        else:
            print(f"{label:8s} (deep):  N/A")


    # 7) Save plots + CSV to SUMMARY_ROOT
    bar_path = os.path.join(SUMMARY_ROOT, "accuracy_bar_2016_70-15-15.png")
    save_accuracy_bar(trad_summary_results, deep_2016_results, bar_path)


    # Confusion matrices for deep models
    for name, fname in [
        ("1D CNN", "cm_cnn.png"),
        ("RNN", "cm_rnn.png"),
        ("Hybrid", "cm_hybrid.png"),
    ]:
        if name in deep_2016_preds:
            out_path = os.path.join(SUMMARY_ROOT, fname)
            save_confusion_matrix(
                y_true=y16_te,
                y_pred=deep_2016_preds[name],
                class_names=class_names,
                title=f"{name} – normalized confusion matrix (2016 TEST)",
                out_path=out_path,
            )

    # SNR curves (deep models)
    snr_path = os.path.join(SUMMARY_ROOT, "snr_curves_2016.png")
    save_snr_curves(
        y_true=y16_te,
        snr_te=snr16_te,
        deep_preds_dict=deep_2016_preds,
        trad_snr_dict=trad_snr_dict,
        out_path=snr_path,
    )


    # Accuracy CSV
    csv_path = os.path.join(SUMMARY_ROOT, "final_accuracies_2016_70-15-15.csv")
    save_accuracy_csv(trad_summary_results, deep_2016_results, csv_path)


    print("\nSummary artifacts written to:", SUMMARY_ROOT)


if __name__ == "__main__":
    main()


Mounted at /content/drive
Using device: cuda

=== Loading original traditional_flat NPZ ===
X_test_flat shape (NPZ): (44000, 256)
y_test shape (NPZ): (44000,)

=== Reloading kNN from /content/drive/MyDrive/amc_runs/final_model_comparison/knn/model.pkl ===
kNN accuracy on ORIGINAL NPZ TEST split: 33.12%

=== Reloading SVM from /content/drive/MyDrive/amc_runs/final_model_comparison/svm/model.pkl ===
SVM accuracy on ORIGINAL NPZ TEST split: 56.25%

=== Reloading RF from /content/drive/MyDrive/amc_runs/final_model_comparison/rf/model.pkl ===
RF accuracy on ORIGINAL NPZ TEST split: 55.90%

=== Traditional models – ORIGINAL NPZ TEST split (%) ===
kNN     : 33.12
SVM     : 56.25
RF      : 55.90

=== Loading RadioML 2016.10A dataset ===
Loaded /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl in (mod, SNR) -> (N, 2, 128) format:
  X shape: (220000, 2, 128)
  y shape: (220000,) dtype: <U6
  snr shape: (220000,) dtype: int64
2016 mods: [np.str_('8PSK'), np.str_('AM-DSB'), np.str_('